# External Wrappers

> 3rd party wrappers for the environments including Pettingzoo, BenchMarl, and RLLIB.

In [ ]:
#| default_exp wrappers.external

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import gymnasium as gym

from gymnasium import spaces
from pettingzoo import ParallelEnv
from typing import Any

from multigrid.envs.base import AgentID, MultiGridEnv


import gymnasium as gym

from ray.rllib.env import MultiAgentEnv
from ray.tune.registry import register_env

from multigrid.envs.base import MultiGridEnv
from multigrid.envs import CONFIGURATIONS
from multigrid.wrappers.base import OneHotObsWrapper


## PettingZoo

Using MultiGrid environments with the PettingZoo ParallelEnv API.

**Usage**

Wrap an environment instance with :class:`.PettingZooWrapper`:

    >>> import gymnasium as gym
    >>> import multigrid.envs
    >>> env = gym.make('MultiGrid-Empty-8x8-v0', agents=2, render_mode='human')

    >>> from multigrid.pettingzoo import PettingZooWrapper
    >>> env = PettingZooWrapper(env)

Wrap an environment class with :func:`.to_pettingzoo_env()`:

    >>> from multigrid.envs import EmptyEnv
    >>> from multigrid.pettingzoo import to_pettingzoo_env
    >>> PZEnv = to_pettingzoo_env(EmptyEnv, metadata={'name': 'empty_v0'})
    >>> env = PZEnv(agents=2, render_mode='human')
"""

In [ ]:
#| export
class PettingZooWrapper(ParallelEnv):
    """
    Wrapper for a ``MultiGridEnv`` environment that implements the
    PettingZoo ``ParallelEnv`` interface.
    """

    def __init__(self, env: MultiGridEnv):
        self.env = env
        self.reset = self.env.reset
        self.step = self.env.step
        self.render = self.env.render
        self.close = self.env.close
        self.metadata = {}

    # @property
    def is_done(self) -> bool:
        return self.env.unwrapped.is_done()
    
    @property
    def agents(self) -> list[AgentID]:
        if self.env.unwrapped.is_done():
            return []
        return [agent.index for agent in self.env.unwrapped.agents if not agent.terminated]

    @property
    def possible_agents(self) -> list[AgentID]:
        return [agent.index for agent in self.env.unwrapped.agents]

    @property
    def observation_spaces(self) -> dict[AgentID, spaces.Space]:
        return dict(self.env.observation_space)

    @property
    def action_spaces(self) -> dict[AgentID, spaces.Space]:
        return dict(self.env.action_space)

    @property
    def render_mode(self) -> str | None:
        return self.env.render_mode

    def observation_space(self, agent_id: AgentID) -> spaces.Space:
        return self.env.observation_space[agent_id]

    def action_space(self, agent_id: AgentID) -> spaces.Space:
        return self.env.action_space[agent_id]


In [ ]:
#| export
def to_pettingzoo_env(
    env_cls: type[MultiGridEnv],
    *wrappers: gym.Wrapper,
    metadata: dict[str, Any] = {}) -> type[ParallelEnv]:
    """
    Convert a ``MultiGridEnv`` environment class to a PettingZoo ``ParallelEnv`` class.

    Note that this is a wrapper around the environment **class**,
    not environment instances.

    Parameters
    ----------
    env_cls : type[MultiGridEnv]
        ``MultiGridEnv`` environment class
    wrappers : gym.Wrapper
        Gym wrappers to apply to the environment
    metadata : dict[str, Any]
        Environment metadata

    Returns
    -------
    pettingzoo_env_cls : type[ParallelEnv]
        PettingZoo ``ParallelEnv`` environment class
    """
    class PettingZooEnv(PettingZooWrapper):
        def __init__(self, *args, **kwargs):
            env = env_cls(*args, **kwargs)
            for wrapper in wrappers:
                env = wrapper(env)
            super().__init__(env)

    PettingZooEnv.__name__ = f"PettingZoo_{env_cls.__name__}"
    PettingZooEnv.metadata = metadata
    return PettingZooEnv


## TorchRL

In [ ]:
#| export
class TorchRLPettingZooWrapper(PettingZooWrapper):
    """
    Extends PettingZooWrapper with string agent IDs required by TorchRL.
    """
    def __init__(self, env: MultiGridEnv):
        # Don't call super().__init__() — we don't want the instance attribute assignments
        self.env = env
        self.metadata = {}

    def _remap_keys(self, d: dict) -> dict:
        """Convert integer agent keys to string agent IDs."""
        return {self._agent_id(k): v for k, v in d.items()}

    def reset(self, seed=None, options=None):
        observations, infos = self.env.reset(seed=seed, options=options)
        obs_remapped = self._remap_keys(observations)
        
        # TorchRL expects {agent_id: {}} for each agent, not just {}
        if not infos:
            infos_remapped = {self._agent_id(i): {} for i in range(len(observations))}
        else:
            infos_remapped = self._remap_keys(infos)
        
        return obs_remapped, infos_remapped
    
    def step(self, actions):
        # Remap string agent IDs back to integer keys for the underlying env
        int_actions = {self._agent_index(k): v for k, v in actions.items()}
        observations, rewards, terminations, truncations, infos = self.env.step(int_actions)
        return (
            self._remap_keys(observations),
            self._remap_keys(rewards),
            self._remap_keys(terminations),
            self._remap_keys(truncations),
            self._remap_keys(infos),
        )

    def _agent_id(self, index: int) -> str:
        return f"agent_{index}"

    def _agent_index(self, agent_id: str) -> int:
        return int(agent_id.split("_")[1])

    @property
    def agents(self) -> list[AgentID]:
        if self.env.unwrapped.is_done():
            return []
        return [self._agent_id(agent.index) 
                for agent in self.env.unwrapped.agents 
                if not agent.terminated]

    @property
    def possible_agents(self) -> list[AgentID]:
        return [self._agent_id(agent.index) 
                for agent in self.env.unwrapped.agents]

    @property
    def observation_spaces(self) -> dict[AgentID, spaces.Space]:
        return {self._agent_id(i): space 
                for i, space in dict(self.env.observation_space).items()}

    @property
    def action_spaces(self) -> dict[AgentID, spaces.Space]:
        return {self._agent_id(i): space 
                for i, space in dict(self.env.action_space).items()}

    def observation_space(self, agent_id: AgentID) -> spaces.Space:
        return self.env.observation_space[self._agent_index(agent_id)]

    def action_space(self, agent_id: AgentID) -> spaces.Space:
        return self.env.action_space[self._agent_index(agent_id)]

## BenchMarl wrapper

## RayLIB

The following provides tools for using MultiGrid environments with
the RLlib MultiAgentEnv API.

**Usage**

Use a specific environment configuration from :mod:`multigrid.envs` by name:

    >>> import multigrid.rllib # registers environment configurations with RLlib
    >>> from ray.rllib.algorithms.ppo import PPOConfig
    >>> algorithm_config = PPOConfig().environment(env='MultiGrid-Empty-8x8-v0')

Wrap an environment instance with :class:`.RLlibWrapper`:

    >>> import gymnasium as gym
    >>> import multigrid.envs
    >>> env = gym.make('MultiGrid-Empty-8x8-v0', agents=2, render_mode='human')

    >>> from multigrid.rllib import RLlibWrapper
    >>> env = RLlibWrapper(env)

Wrap an environment class with :func:`.to_rllib_env()`:

    >>> from multigrid.envs import EmptyEnv
    >>> from multigrid.rllib import to_rllib_env
    >>> MyEnv = to_rllib_env(EmptyEnv, default_config={'size': 8})
    >>> config = {'agents': 2, 'render_mode': 'human'}
    >>> env = MyEnv(config)

In [ ]:
#| export

class RLlibWrapper(MultiAgentEnv):
    """
    Wrapper for a ``MultiGridEnv`` environment that implements the
    RLlib ``MultiAgentEnv`` interface.
    """

    def __init__(self, env: MultiGridEnv):
        super().__init__()
        self.env = env
        self.agents = list(range(len(env.unwrapped.agents)))
        self.possible_agents = self.agents[:]

    def reset(self, *args, **kwargs):
        return self.env.reset(*args, **kwargs)

    def step(self, *args, **kwargs):
        obs, rewards, terminations, truncations, infos = self.env.step(*args, **kwargs)
        terminations['__all__'] = all(terminations.values())
        truncations['__all__'] = all(truncations.values())
        return obs, rewards, terminations, truncations, infos

    def get_observation_space(self, agent_index: int):
        return self.env.unwrapped.agents[agent_index].observation_space

    def get_action_space(self, agent_index: int):
        return self.env.unwrapped.agents[agent_index].action_space
    

In [ ]:
#| export

def to_rllib_env(
    env_cls: type[MultiGridEnv],
    *wrappers: gym.Wrapper,
    default_config: dict = {}) -> type[MultiAgentEnv]:
    """
    Convert a ``MultiGridEnv`` environment class to an RLLib ``MultiAgentEnv`` class.

    Note that this is a wrapper around the environment **class**,
    not environment instances.

    Parameters
    ----------
    env_cls : type[MultiGridEnv]
        ``MultiGridEnv`` environment class
    wrappers : gym.Wrapper
        Gym wrappers to apply to the environment
    default_config : dict
        Default configuration for the environment

    Returns
    -------
    rllib_env_cls : type[MultiAgentEnv]
        RLlib ``MultiAgentEnv`` environment class
    """
    class RLlibEnv(RLlibWrapper):
        def __init__(self, config: dict = {}):
            config = {**default_config, **config}
            env = env_cls(**config)
            for wrapper in wrappers:
                env = wrapper(env)
            super().__init__(env)

    RLlibEnv.__name__ = f"RLlib_{env_cls.__name__}"
    return RLlibEnv



In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()